# Estimated Pathway Activity
Compute the estimated pathway activity (with OVCA data)

tutorial: https://decoupler.readthedocs.io/en/latest/notebooks/spatial/rna_visium.html

In [13]:
import warnings

import scanpy as sc
import decoupler as dc
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
sc.set_figure_params(figsize=(3, 3), frameon=False)

In [3]:
adata = sc.read_h5ad('/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/adata_annotated.h5ad')
adata

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 113277 × 19059
    obs: 'orig.ident', 'nCount_Spatial', 'nFeature_Spatial', 'Region', 'alignment_clusters', 'spot_class', 'first_type', 'second_type', 'min_score', 'singlet_score', 'seurat_cluster.sketched', 'seurat_clusters', 'metaclusters', 'AnatomicalSite', 'WoundSite', 'seurat_cluster.projected', 'seurat_cluster.projected.score', 'leverage.score', 'banksy_cluster_0.8', 'banksy_cluster_0.5', 'barcode'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'

In [4]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.layers["norm"] = adata.X.copy()

In [ ]:
progeny = dc.op.progeny(organism='human')

# check overlap between progeny genes and your mouse genes
progeny_genes = set(progeny['target'].unique())
mouse_genes = set(adata.var_names)
overlap = progeny_genes & mouse_genes
print(f"PROGENy genes: {len(progeny_genes)}")
print(f"overlap with your mouse genes: {len(overlap)}")
print(f"overlap %: {len(overlap)/len(progeny_genes)*100:.1f}%")

PROGENy genes: 17610
overlap with your mouse genes: 15
overlap %: 0.1%


In [11]:
# try uppercasing mouse genes
mouse_genes_upper = {g.upper() for g in mouse_genes}
overlap_upper = progeny_genes & mouse_genes_upper
print(f"overlap after uppercasing: {len(overlap_upper)}")
print(f"overlap %: {len(overlap_upper)/len(progeny_genes)*100:.1f}%")

adata.var_names = adata.var_names.str.upper()

overlap after uppercasing: 14570
overlap %: 82.7%


In [14]:
adata.var_names_make_unique()

# compute pathway scores
chunk_size = 5000
results = []

for i in range(0, adata.n_obs, chunk_size):
    print(f"Processing cells {i} to {min(i+chunk_size, adata.n_obs)}...")
    chunk = adata[i:i+chunk_size].copy()
    chunk.var_names_make_unique()
    dc.mt.ulm(data=chunk, net=progeny)
    results.append(chunk.obsm['score_ulm'])

pathway_matrix = pd.concat(results)
pathway_matrix.index = adata.obs_names

pathways = ['EGFR', 'Androgen', 'Estrogen', 'JAK-STAT', 'VEGF', 'MAPK', 'PI3K', 'TGFb', 'NFkB', 'TNFa']
pathway_matrix[pathways].to_parquet('/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/pathway_activation.parquet')
print("saved")

Processing cells 0 to 5000...
Processing cells 5000 to 10000...
Processing cells 10000 to 15000...
Processing cells 15000 to 20000...
Processing cells 20000 to 25000...
Processing cells 25000 to 30000...
Processing cells 30000 to 35000...
Processing cells 35000 to 40000...
Processing cells 40000 to 45000...
Processing cells 45000 to 50000...
Processing cells 50000 to 55000...
Processing cells 55000 to 60000...
Processing cells 60000 to 65000...
Processing cells 65000 to 70000...
Processing cells 70000 to 75000...
Processing cells 75000 to 80000...
Processing cells 80000 to 85000...
Processing cells 85000 to 90000...
Processing cells 90000 to 95000...
Processing cells 95000 to 100000...
Processing cells 100000 to 105000...
Processing cells 105000 to 110000...
Processing cells 110000 to 113277...
saved


In [15]:
print(adata.obsm['score_ulm'].columns.tolist())

KeyError: 'score_ulm'

In [16]:
progeny_pathways = ['EGFR', 'Androgen', 'Estrogen', 'JAK-STAT', 'VEGF', 'MAPK', 'PI3K', 'TGFb', 'NFkB', 'TNFa'] # TGFA is missing (noted pathway in paper)

pathway_matrix = adata.obsm['score_ulm'][progeny_pathways]
pathway_matrix.index = adata.obs_names

pathway_matrix.to_parquet('/insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_tutorial/pathway_activation.parquet')
print(pathway_matrix.shape)
print(pathway_matrix.head())

KeyError: 'score_ulm'